In [1]:
import os  # Used to navigate through folders and find files
import numpy as np  # Used to handle large matrices and mathematical arrays
from tensorflow.keras.preprocessing.sequence import pad_sequences  # Used to make all videos equal length
from sklearn.model_selection import train_test_split  # Used to separate training data from testing data
from tensorflow.keras.utils import to_categorical  # Used to format your gesture labels

print("Libraries successfully loaded!")

Libraries successfully loaded!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Note: If you named your zip file something else, change 'archive.zip' below
!unzip -q "/content/drive/MyDrive/archive.zip" -d "/content/dhg_data"
print("Unzipping complete!")

Unzipping complete!


In [4]:
import os
import numpy as np

# We point directly to where Colab just unzipped the files
data_dir = '/content/dhg_data'

X_raw = []
y_raw = []

print("Opening folders and reading files... This will be incredibly fast now.")

for folder_name in sorted(os.listdir(data_dir)):
    if folder_name.startswith('gesture_'):

        gesture_label = int(folder_name.split('_')[1]) - 1
        gesture_folder_path = os.path.join(data_dir, folder_name)

        for root, subfolders, files in os.walk(gesture_folder_path):
            for file in files:
                if file == 'skeleton_world.txt':
                    full_file_path = os.path.join(root, file)

                    try:
                        video_matrix = np.loadtxt(full_file_path)

                        if video_matrix.ndim == 2 and video_matrix.shape[0] > 0:
                            X_raw.append(video_matrix)
                            y_raw.append(gesture_label)
                    except:
                        pass

y_raw = np.array(y_raw)
print(f"Success! We found and loaded {len(X_raw)} gesture videos.")

Opening folders and reading files... This will be incredibly fast now.
Success! We found and loaded 0 gesture videos.


In [5]:
import os
print(os.listdir('/content/dhg_data'))

['train8.csv', 'val - 1.csv', 'val -2.csv', 'train - 3.csv', 'train - 1.csv', 'train - main2.csv', 'train - 2.csv', 'DHG2016', 'train - 4.csv', 'train -main.csv', 'val - 3.csv', 'train.csv', 'val - 4.csv', 'val - main2.csv', 'val8.csv', 'val.csv']


In [6]:
import os
import numpy as np

# We updated the path to point inside the DHG2016 folder!
data_dir = '/content/dhg_data/DHG2016'

X_raw = []
y_raw = []

print("Opening folders and reading files... Here we go!")

for folder_name in sorted(os.listdir(data_dir)):
    # We only care about folders that start with "gesture_"
    if folder_name.startswith('gesture_'):

        # Figure out the gesture number (e.g., "gesture_1" becomes class 0)
        gesture_label = int(folder_name.split('_')[1]) - 1
        gesture_folder_path = os.path.join(data_dir, folder_name)

        # Dig into every single sub-folder automatically
        for root, subfolders, files in os.walk(gesture_folder_path):
            for file in files:

                # If we find the specific 3D tracking file...
                if file == 'skeleton_world.txt':
                    full_file_path = os.path.join(root, file)

                    try:
                        # Read the text file and turn it into a math matrix
                        video_matrix = np.loadtxt(full_file_path)

                        # If the file has data inside, save it to our lists
                        if video_matrix.ndim == 2 and video_matrix.shape[0] > 0:
                            X_raw.append(video_matrix)
                            y_raw.append(gesture_label)
                    except:
                        pass

# Convert our list of labels into a solid math array
y_raw = np.array(y_raw)

print(f"Success! We found and loaded {len(X_raw)} gesture videos.")

Opening folders and reading files... Here we go!
Success! We found and loaded 2800 gesture videos.


In [7]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Step 3: Starting Zero-Gravity Normalization...")
X_normalized = []

# 1. THE NORMALIZATION LOOP
for video in X_raw:
    # Create a blank slate to store our new, clean coordinates
    clean_video = np.zeros_like(video)

    # Loop through every single frame in this specific video
    for frame_idx in range(video.shape[0]):
        # The first 3 numbers (columns 0, 1, 2) are the X, Y, Z of the Palm Center
        palm_xyz = video[frame_idx, 0:3]

        # Loop through all 22 joints and shift them
        for joint_idx in range(22):
            start_col = joint_idx * 3
            end_col = start_col + 3

            # Subtract the palm position from this joint's position
            clean_video[frame_idx, start_col:end_col] = video[frame_idx, start_col:end_col] - palm_xyz

    X_normalized.append(clean_video)

print("Normalization complete! Now standardizing video lengths...")

# 2. THE PADDING PROCESS
MAX_FRAMES = 60

# pad_sequences automatically handles the math of resizing our videos
X_final = pad_sequences(
    X_normalized,
    maxlen=MAX_FRAMES,
    dtype='float32',
    padding='post',      # Add dummy frames (zeros) at the END if the video is too short
    truncating='post'    # Cut off extra frames at the END if the video is too long
)

print(f"Preprocessing Complete! Your final data shape is: {X_final.shape}")

Step 3: Starting Zero-Gravity Normalization...
Normalization complete! Now standardizing video lengths...
Preprocessing Complete! Your final data shape is: (2800, 60, 66)


In [8]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

print("Formatting data for the neural network...")

# Convert gesture labels (0-13) into 14-column binary vectors
y_final = to_categorical(y_raw, num_classes=14)

# Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X_final, # Your padded (2800, 60, 66) array from Step 3
    y_final,
    test_size=0.2,
    random_state=42,
    stratify=y_raw # Ensures every gesture has an equal amount of testing data
)

print(f"Training set ready: {X_train.shape[0]} sequences.")
print(f"Testing set ready: {X_test.shape[0]} sequences.")

Formatting data for the neural network...
Training set ready: 2240 sequences.
Testing set ready: 560 sequences.


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, GRU, Dense, Dropout, BatchNormalization

print("Constructing the Gesture Recognition Model...")

model = Sequential([
    # 1. Masking Layer: Tells the AI to ignore all the zero-padding we added to short videos
    Masking(mask_value=0.0, input_shape=(60, 66)),

    # 2. First Memory Block: Learns the primary movements of the 22 joints
    GRU(64, return_sequences=True),
    BatchNormalization(), # Stabilizes learning
    Dropout(0.3),         # Prevents the AI from memorizing the data (overfitting)

    # 3. Second Memory Block: Learns the complex sequence of the gesture
    GRU(64),
    BatchNormalization(),
    Dropout(0.3),

    # 4. Decision Layer: Condenses the learned patterns
    Dense(32, activation='relu'),

    # 5. Output Layer: 14 nodes, one for each gesture class
    Dense(14, activation='softmax')
])

# Compile the model with the Adam optimizer
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Constructing the Gesture Recognition Model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 60, 66)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 60, 64)         │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │           462 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,358 (208.43 KB)

 Trainable params: 53,102 (207.43 KB)

 Non-trainable params: 256 (1.00 KB)

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

# THE SAFETY NET: Early Stopping
# This monitors the AI. If the AI stops getting better at understanding
# the test data for 15 rounds (patience=15), it will automatically stop
# the training so it doesn't waste time or break the model.
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("Initiating model training sequence... Watch the numbers closely!")

history = model.fit(
    X_train, y_train,
    epochs=150,               # Maximum number of training loops (Epochs)
    batch_size=32,            # The AI looks at 32 videos at a time before updating its brain
    validation_split=0.2,     # It holds back 20% of the training data to test itself at the end of every round
    callbacks=[early_stop]    # Attaches our safety net
)

Initiating model training sequence... Watch the numbers closely!
Epoch 1/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.1412 - loss: 2.6987 - val_accuracy: 0.0692 - val_loss: 2.6382
Epoch 2/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.2109 - loss: 2.4529 - val_accuracy: 0.0692 - val_loss: 2.6399
Epoch 3/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2561 - loss: 2.2776 - val_accuracy: 0.0692 - val_loss: 2.6406
Epoch 4/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2863 - loss: 2.1991 - val_accuracy: 0.0737 - val_loss: 2.6453
Epoch 5/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2919 - loss: 2.1641 - val_accuracy: 0.0982 - val_loss: 2.6276
Epoch 6/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3237 - loss: 2.0730 - val_accuracy: 0.0982 - val_loss: 2.7333
Epoch 7/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3281 - loss: 2.0350 - val_accuracy: 0.2232 - val_loss: 2.4164
Epoch 8/150
56/56 ━━━━━━━━━━━━━━━━━━━━

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, GRU, Dense, Dropout, BatchNormalization

print("Constructing the Gesture Recognition Model...")

model = Sequential([
    # 1. Masking Layer: Tells the AI to ignore all the zero-padding we added to short videos
    Masking(mask_value=0.0, input_shape=(60, 66)),

    # 2. First Memory Block: Learns the primary movements of the 22 joints
    GRU(64, return_sequences=True),
    BatchNormalization(), # Stabilizes learning
    Dropout(0.3),         # Prevents the AI from memorizing the data (overfitting)

    # 3. Second Memory Block: Learns the complex sequence of the gesture
    GRU(64),
    BatchNormalization(),
    Dropout(0.5),

    # 4. Decision Layer: Condenses the learned patterns
    Dense(32, activation='relu'),

    # 5. Output Layer: 14 nodes, one for each gesture class
    Dense(14, activation='softmax')
])

# Compile the model with the Adam optimizer
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Constructing the Gesture Recognition Model...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_1 (Masking)             │ (None, 60, 66)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 60, 64)         │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 14)             │           462 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,358 (208.43 KB)

 Trainable params: 53,102 (207.43 KB)

 Non-trainable params: 256 (1.00 KB)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

# The Safety Net
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("Initiating model training sequence (Round 2)... Watch the numbers closely!")

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Initiating model training sequence (Round 2)... Watch the numbers closely!
Epoch 1/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.1222 - loss: 2.9791 - val_accuracy: 0.1406 - val_loss: 2.6243
Epoch 2/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.1775 - loss: 2.5661 - val_accuracy: 0.1473 - val_loss: 2.6084
Epoch 3/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.2215 - loss: 2.4463 - val_accuracy: 0.1585 - val_loss: 2.5878
Epoch 4/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2260 - loss: 2.3985 - val_accuracy: 0.1786 - val_loss: 2.5587
Epoch 5/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2561 - loss: 2.2798 - val_accuracy: 0.2232 - val_loss: 2.5226
Epoch 6/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2891 - loss: 2.2473 - val_accuracy: 0.1987 - val_loss: 2.5013
Epoch 7/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2952 - loss: 2.1856 - val_accuracy: 0.2321 - val_loss: 2.3941
Epoch 8/150
56/56 ━━━━━━━━━━━

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, GRU, Dense, Dropout, BatchNormalization

print("Constructing the Upgraded V3 Spacecraft AI...")

model = Sequential([
    Masking(mask_value=0.0, input_shape=(60, 66)),

    # UPGRADE: Increased from 64 to 128 memory nodes
    GRU(128, return_sequences=True),
    BatchNormalization(),
    Dropout(0.4), # Keeping the hard-mode training!

    # UPGRADE: Increased from 64 to 128 memory nodes
    GRU(128),
    BatchNormalization(),
    Dropout(0.4),

    Dense(32, activation='relu'),
    Dense(14, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Brain successfully wiped and upgraded. Ready for training.")

Constructing the Upgraded V3 Spacecraft AI...
Brain successfully wiped and upgraded. Ready for training.


In [14]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("Initiating Training Sequence V3... Watch the val_accuracy!")

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Initiating Training Sequence V3... Watch the val_accuracy!
Epoch 1/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.1551 - loss: 2.7514 - val_accuracy: 0.0580 - val_loss: 2.6316
Epoch 2/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2360 - loss: 2.4300 - val_accuracy: 0.0625 - val_loss: 2.6203
Epoch 3/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2556 - loss: 2.3478 - val_accuracy: 0.0781 - val_loss: 2.6131
Epoch 4/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3025 - loss: 2.2009 - val_accuracy: 0.0826 - val_loss: 2.6096
Epoch 5/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3242 - loss: 2.1319 - val_accuracy: 0.1027 - val_loss: 2.6398
Epoch 6/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.3348 - loss: 2.0568 - val_accuracy: 0.1272 - val_loss: 2.6057
Epoch 7/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.3571 - loss: 1.9772 - val_accuracy: 0.1161 - val_loss: 2.5852
Epoch 8/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 22m

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, GRU, Dense, Dropout, BatchNormalization

print("Constructing the Final V4 Spacecraft AI (128 Nodes + 0.5 Dropout)...")

model = Sequential([
    Masking(mask_value=0.0, input_shape=(60, 66)),

    # Combined Power: Large 128-node capacity + Strict 0.5 Dropout
    GRU(128, return_sequences=True),
    BatchNormalization(),
    Dropout(0.5),

    GRU(128),
    BatchNormalization(),
    Dropout(0.5),

    Dense(32, activation='relu'),
    Dense(14, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Brain initialized. Ready for the final training run.")

Constructing the Final V4 Spacecraft AI (128 Nodes + 0.5 Dropout)...
Brain initialized. Ready for the final training run.


In [16]:
from tensorflow.keras.callbacks import EarlyStopping

# The Safety Net
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("Initiating Final V4 Training Sequence...")
print("Goal: High Capacity (128 nodes) + Maximum Stability (0.5 Dropout)")

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Initiating Final V4 Training Sequence...
Goal: High Capacity (128 nodes) + Maximum Stability (0.5 Dropout)
Epoch 1/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.1362 - loss: 2.9191 - val_accuracy: 0.0826 - val_loss: 2.6267
Epoch 2/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.2227 - loss: 2.5326 - val_accuracy: 0.0804 - val_loss: 2.6264
Epoch 3/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.2260 - loss: 2.4540 - val_accuracy: 0.1250 - val_loss: 2.6121
Epoch 4/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2584 - loss: 2.3261 - val_accuracy: 0.1384 - val_loss: 2.5918
Epoch 5/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2645 - loss: 2.3038 - val_accuracy: 0.1473 - val_loss: 2.5538
Epoch 6/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3103 - loss: 2.1368 - val_accuracy: 0.1406 - val_loss: 2.5208
Epoch 7/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3086 - loss: 2.1168 - val_accuracy: 0.1607 - val_loss: 2.46

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, GRU, Dense, Dropout, BatchNormalization

print("Constructing the Official Spacecraft AI (Round 2 Architecture)...")

model = Sequential([
    Masking(mask_value=0.0, input_shape=(60, 66)),

    # The optimal balance of capacity and stability
    GRU(64, return_sequences=True),
    BatchNormalization(),
    Dropout(0.5),

    GRU(64),
    BatchNormalization(),
    Dropout(0.5),

    Dense(32, activation='relu'),
    Dense(14, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Brain locked and loaded.")

Constructing the Official Spacecraft AI (Round 2 Architecture)...
Brain locked and loaded.


In [18]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("Initiating Final Mission Training...")

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Initiating Final Mission Training...
Epoch 1/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.1060 - loss: 3.0456 - val_accuracy: 0.0871 - val_loss: 2.6364
Epoch 2/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.1680 - loss: 2.6511 - val_accuracy: 0.0982 - val_loss: 2.6281
Epoch 3/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.1953 - loss: 2.5218 - val_accuracy: 0.0848 - val_loss: 2.6223
Epoch 4/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2360 - loss: 2.4061 - val_accuracy: 0.1585 - val_loss: 2.5922
Epoch 5/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.2506 - loss: 2.3578 - val_accuracy: 0.1384 - val_loss: 2.5654
Epoch 6/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.2628 - loss: 2.2849 - val_accuracy: 0.1652 - val_loss: 2.5218
Epoch 7/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.2684 - loss: 2.2740 - val_accuracy: 0.1853 - val_loss: 2.4727
Epoch 8/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.2

In [19]:
print("Running final diagnostics on unseen test data...")

# 1. Final Accuracy Check on the Test Set
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("\n" + "="*30)
print("🚀 OFFICIAL MISSION CLEARANCE 🚀")
print("="*30)
print(f"Final Model Accuracy: {test_accuracy * 100:.2f}%")

# 2. Real-Time Latency Speed Test
import time

# Warm up the GPU first
_ = model.predict(X_test[0:1], verbose=0)

# Time exactly how long a single gesture prediction takes
start_time = time.time()
sample_prediction = model.predict(X_test[0:1], verbose=0)
end_time = time.time()

latency = (end_time - start_time) * 1000
print(f"Inference Latency: {latency:.2f} milliseconds per gesture.")

if latency < 50:
    print("Latency Status: CLEAR FOR REAL-TIME DEPLOYMENT")
else:
    print("Latency Status: WARNING - SYSTEM TOO SLOW")

# 3. Export the AI Brain
model_filename = 'space_gesture_model.keras'
model.save(model_filename)

print(f"\nModel successfully exported as: {model_filename}")
print("Check the folder icon on the left menu in Colab to download your AI!")

Running final diagnostics on unseen test data...
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5214 - loss: 1.4828

🚀 OFFICIAL MISSION CLEARANCE 🚀
Final Model Accuracy: 52.14%
Inference Latency: 106.83 milliseconds per gesture.
Latency Status: WARNING - SYSTEM TOO SLOW

Model successfully exported as: space_gesture_model.keras
Check the folder icon on the left menu in Colab to download your AI!
